# Swiss Legal Retrieval — Google Colab Setup

執行順序：**Cell 1 → 2 → 3 → 4 → 5**，之後就可以直接跑 pipeline。

| Cell | 功能 |
|------|------|
| 1 | Clone repo + 安裝套件 |
| 2 | 掛載 Google Drive，設定 DATA_DIR / INDEX_DIR |
| 3 | 檢查 GPU |
| 4 | 設定 ANTHROPIC_API_KEY |
| 5 | Index guard：已有則載入，沒有則建立並存到 Drive |

In [ ]:
# ── Cell 1: Clone repo + 安裝套件 ────────────────────────────────────────────
import subprocess, sys

GITHUB_REPO = "https://github.com/YOUR_USERNAME/swiss-legal-retrieval.git"  # ← 改成你的 repo
REPO_DIR    = "/content/swiss-legal-retrieval"

# Clone（已存在則跳過，避免重複執行時出錯）
import os
if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print(f"Repo already exists at {REPO_DIR}, pulling latest...")
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
sys.path.insert(0, f"{REPO_DIR}/src")

# 安裝套件
!pip install -q \
    rank-bm25 \
    sentence-transformers \
    faiss-gpu \
    anthropic \
    python-dotenv \
    pyyaml \
    pandas

print("\n✓ 套件安裝完成")

In [ ]:
# ── Cell 2: 掛載 Google Drive，設定路徑 ──────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive", force_remount=False)

# ↓ 修改成你在 Drive 裡存放資料的路徑
DRIVE_ROOT  = Path("/content/drive/MyDrive/swiss-legal-retrieval")
DATA_DIR    = DRIVE_ROOT / "data"
INDEX_DIR   = DRIVE_ROOT / "indices"
OUTPUT_DIR  = Path("/content/output")   # 中間結果放 /content，不佔 Drive 空間

DATA_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 確認資料存在
LAWS_PATH   = DATA_DIR / "federal_laws.jsonl"
COURTS_PATH = DATA_DIR / "court_decisions.jsonl"
TEST_CSV    = DATA_DIR / "test.csv"
TRAIN_CSV   = DATA_DIR / "train.csv"

missing = [p for p in [LAWS_PATH, COURTS_PATH, TEST_CSV] if not p.exists()]
if missing:
    print("[WARN] 以下資料檔案不存在，請先上傳到 Drive：")
    for p in missing:
        print(f"  {p}")
    print("\n下載指令（在 Kaggle API 設定後）：")
    print("  kaggle competitions download -c llm-agentic-legal-information-retrieval -p", DATA_DIR)
else:
    print("✓ 資料目錄：", DATA_DIR)
    print("✓ Index 目錄：", INDEX_DIR)
    for p in [LAWS_PATH, COURTS_PATH, TEST_CSV, TRAIN_CSV]:
        size_mb = p.stat().st_size / 1e6 if p.exists() else 0
        status = f"{size_mb:.1f} MB" if p.exists() else "NOT FOUND"
        print(f"  {p.name:35s} {status}")

In [ ]:
# ── Cell 3: 檢查 GPU ──────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"✓ GPU 可用：{gpu_name}  ({vram_gb:.1f} GB VRAM)")
    DEVICE = "cuda"
else:
    print("[WARN] GPU 不可用，使用 CPU（dense encoding 會很慢）")
    print("  Runtime → 變更執行階段類型 → GPU")
    DEVICE = "cpu"

# faiss：Colab GPU 環境用 faiss-gpu，否則用 faiss-cpu
try:
    import faiss
    gpu_res = faiss.StandardGpuResources() if DEVICE == "cuda" else None
    print(f"✓ FAISS 版本：{faiss.__version__}  ({'GPU' if DEVICE == 'cuda' else 'CPU'} mode)")
except ImportError:
    print("[ERROR] faiss 未安裝，請重新執行 Cell 1")

In [ ]:
# ── Cell 4: 設定 ANTHROPIC_API_KEY（從 Colab Secrets 讀取）────────────────────
#
# 設定方式：左側欄 🔑 圖示 → 新增 secret
#   Name:  ANTHROPIC_API_KEY
#   Value: sk-ant-...
#   開啟「讓這個筆記本存取」

import os

try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
    if api_key:
        os.environ["ANTHROPIC_API_KEY"] = api_key
        masked = api_key[:10] + "..." + api_key[-4:]
        print(f"✓ ANTHROPIC_API_KEY 已載入（{masked}）")
    else:
        print("[WARN] ANTHROPIC_API_KEY secret 為空，Claude reranking 將無法使用")
except Exception:
    # 本地或非 Colab 環境：嘗試從 .env 讀取
    from dotenv import load_dotenv
    load_dotenv()
    if os.environ.get("ANTHROPIC_API_KEY"):
        print("✓ ANTHROPIC_API_KEY 已從 .env 載入")
    else:
        print("[WARN] 找不到 ANTHROPIC_API_KEY，Claude reranking 將無法使用")
        print("  請到 Colab 左側欄 🔑 新增 ANTHROPIC_API_KEY secret")

# 驗證 API key 是否有效
if os.environ.get("ANTHROPIC_API_KEY"):
    try:
        import anthropic
        client = anthropic.Anthropic()
        # 用最小的請求測試連線
        client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=5,
            messages=[{"role": "user", "content": "hi"}]
        )
        print("✓ Anthropic API 連線正常")
    except Exception as e:
        print(f"[ERROR] API 驗證失敗：{e}")

In [ ]:
# ── Cell 5: Index guard — 已有則載入，沒有則建立並存到 Drive ──────────────────
import json, pickle
import numpy as np

DENSE_MODEL  = "intfloat/multilingual-e5-large"
PASSAGE_PFX  = "passage: "
ENCODE_BATCH = 64

BM25_LAWS_PATH     = INDEX_DIR / "bm25_laws.pkl"
BM25_COURTS_PATH   = INDEX_DIR / "bm25_courts.pkl"
FAISS_LAWS_PATH    = INDEX_DIR / "dense_laws.faiss"
FAISS_COURTS_PATH  = INDEX_DIR / "dense_courts.faiss"
META_LAWS_PATH     = INDEX_DIR / "dense_laws_meta.jsonl"
META_COURTS_PATH   = INDEX_DIR / "dense_courts_meta.jsonl"

sys.path.insert(0, f"{REPO_DIR}/src")
from omnilex.retrieval.bm25_index import BM25Index, load_jsonl_corpus

# ── 載入語料庫 ────────────────────────────────────────────────────────────────
print("載入語料庫...")
laws   = load_jsonl_corpus(str(LAWS_PATH))
courts = load_jsonl_corpus(str(COURTS_PATH))
print(f"  Laws: {len(laws)}, Courts: {len(courts)}")

# ── BM25 guard ────────────────────────────────────────────────────────────────
if BM25_LAWS_PATH.exists() and BM25_COURTS_PATH.exists():
    print("\n[BM25] Index 已存在，直接載入...")
    bm25_laws   = BM25Index(); bm25_laws.load(str(BM25_LAWS_PATH))
    bm25_courts = BM25Index(); bm25_courts.load(str(BM25_COURTS_PATH))
    print(f"  ✓ Laws BM25: {len(bm25_laws.documents)} docs")
    print(f"  ✓ Courts BM25: {len(bm25_courts.documents)} docs")
else:
    print("\n[BM25] 建立 index...")
    bm25_laws   = BM25Index(text_field="text", citation_field="citation")
    bm25_courts = BM25Index(text_field="text", citation_field="citation")
    bm25_laws.build(laws)
    bm25_courts.build(courts)
    bm25_laws.save(str(BM25_LAWS_PATH))
    bm25_courts.save(str(BM25_COURTS_PATH))
    print(f"  ✓ BM25 index 已儲存到 Drive")

# ── Dense (FAISS) guard ───────────────────────────────────────────────────────
import faiss

if FAISS_LAWS_PATH.exists() and FAISS_COURTS_PATH.exists():
    print("\n[Dense] Index 已存在，直接載入...")
    faiss_laws   = faiss.read_index(str(FAISS_LAWS_PATH))
    faiss_courts = faiss.read_index(str(FAISS_COURTS_PATH))
    laws_meta    = load_jsonl_corpus(str(META_LAWS_PATH))
    courts_meta  = load_jsonl_corpus(str(META_COURTS_PATH))

    # GPU 加速（若有）
    if DEVICE == "cuda" and gpu_res:
        faiss_laws   = faiss.index_cpu_to_gpu(gpu_res, 0, faiss_laws)
        faiss_courts = faiss.index_cpu_to_gpu(gpu_res, 0, faiss_courts)
        print("  ✓ FAISS index 移到 GPU")

    print(f"  ✓ Laws FAISS: {faiss_laws.ntotal} vectors")
    print(f"  ✓ Courts FAISS: {faiss_courts.ntotal} vectors")
    dense_model = None   # 不需要 encode 新文件，懶載入
else:
    print("\n[Dense] 建立 index（需要幾分鐘）...")
    from sentence_transformers import SentenceTransformer
    dense_model = SentenceTransformer(DENSE_MODEL, device=DEVICE)

    def build_faiss(docs, save_path, meta_path):
        texts = [PASSAGE_PFX + (d.get("text") or d.get("regeste") or "") for d in docs]
        embs  = dense_model.encode(
            texts, batch_size=ENCODE_BATCH,
            normalize_embeddings=True, show_progress_bar=True
        ).astype("float32")
        dim   = embs.shape[1]
        idx   = faiss.IndexFlatIP(dim)
        idx.add(embs)
        faiss.write_index(idx, str(save_path))
        with open(meta_path, "w", encoding="utf-8") as f:
            for d in docs: f.write(json.dumps(d, ensure_ascii=False) + "\n")
        return idx

    print("  Encoding laws...")
    faiss_laws  = build_faiss(laws,   FAISS_LAWS_PATH,   META_LAWS_PATH)
    print("  Encoding courts...")
    faiss_courts = build_faiss(courts, FAISS_COURTS_PATH, META_COURTS_PATH)

    laws_meta   = laws
    courts_meta = courts

    if DEVICE == "cuda" and gpu_res:
        faiss_laws   = faiss.index_cpu_to_gpu(gpu_res, 0, faiss_laws)
        faiss_courts = faiss.index_cpu_to_gpu(gpu_res, 0, faiss_courts)

    print(f"  ✓ Dense index 已儲存到 Drive")

print("\n=== 環境就緒，可以開始 retrieve ===")